In [13]:
import os

os.environ["KERAS_BACKEND"] = "torch"  # Or "jax" / "torch"

from dataclasses import dataclass
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path

fhvhv = True

In [14]:
import torch

# Check if CUDA is available
is_available = torch.cuda.is_available()
if is_available:
    # Print the name of the AMD GPU
    print(f"Device name: {torch.cuda.get_device_name(0)}")
    
    # Perform a test tensor allocation on the GPU
    t = torch.tensor([1, 2, 3], device='cuda')
    print(f"Test tensor successfully placed on: {t.device}")
else:
    print("GPU not detected. PyTorch will run on CPU.")


Device name: AMD Radeon RX 9070 XT
Test tensor successfully placed on: cuda:0


In [15]:
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
projectRoot = Path.cwd().parents[2]
dataPath = Path(projectRoot, 
                "data/raw/") # Replace with the path of your data
taxiPath = Path(dataPath, "tlc_trip_record/yellow/")
weatherPath = Path(dataPath, "weather/")
lookUpPath = Path(projectRoot, "data/NYC_Taxi_Zones_20251203.csv")

In [16]:
from dataclasses import dataclass
@dataclass
class dates:
    start_year: int
    start_month: int
    end_year: int
    end_month: int

In [17]:
from dataset.taxi_demand import TaxiDemand, FHVHVTaxiDemand
def get_taxiDemand(date: dates):
    if fhvhv:
        return FHVHVTaxiDemand(date.start_year, date.start_month, date.end_year, date.end_month)
    else:
        return TaxiDemand(date.start_year, date.start_month, date.end_year, date.end_month)

In [18]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


train_dates = dates(2025,1,2025,7)
train = get_taxiDemand(train_dates)

input_steps = 12
target_index = 0
feature_list = ['pickups', 'precipitation', 'wind_speed_10m', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']

data_train = train.prepare_dataset(feature_list)
X_train, Y_train = train.construct_dataset(data_train, input_steps, target_index)

In [20]:
from keras.models import load_model
from layers.stgcn import STGCNBlock
from layers.adaptive_adjacency import AdaptiveAdjacency
dense_model = load_model(
     "dense.keras", 
    compile=False
)
rnn_model = load_model(
     "rnn.keras", 
    compile=False
)
best_model = load_model(
    "stgcn_fhvh.keras", 
    compile=False
)

In [24]:
best_model.summary()

Model: "STGCN"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ features            │ (None, 12, 263,   │          0 │ -                 │
│ (InputLayer)        │ 7)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_projection    │ (None, 12, 263,   │      2,048 │ features[0][0]    │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ adaptive_adj        │ (263, 263)        │     16,832 │ features[0][0]    │
│ (AdaptiveAdjacency) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stgcn_block_1       │ (None, 12, 263,   │    854,528 │ input_projection… │
│ (STGCNBlock)        │ 256)              │            │ adaptive_adj[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stgcn_block_2       │ (None, 12, 263,   │    854,528 │ stgcn_block_1[0]… │
│ (STGCNBlock)        │ 256)              │            │ adaptive_adj[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 263, 256)  │          0 │ stgcn_block_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 263, 1)    │        257 │ get_item[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,728,193 (6.59 MB)

 Trainable params: 1,728,193 (6.59 MB)

 Non-trainable params: 0 (0.00 B)

In [21]:
seq_len = input_steps
train_hours = pd.date_range(start=train.start_date, end=train.end_date, freq="1h", inclusive = "left")[seq_len:].hour.values
historical_avg = {}
for h in range(24):
    mask = (train_hours == h)
    if np.any(mask):
        # Mean across the time dimension for this specific hour
        # Resulting shape: (N, num_targets)
        historical_avg[h] = Y_train[mask].mean(axis=0)
    else:
        # Fallback in case an hour is completely missing in train data
        historical_avg[h] = np.zeros_like(Y_train[0])

In [22]:
def result(start_year, start_month, end_year, end_month):
    test_dates = dates(start_year,start_month,end_year,end_month)
    test = get_taxiDemand(test_dates)
    data_test = test.prepare_dataset(feature_list)
    #data_test = data_train
    X_test, Y_test = test.construct_dataset(data_test, input_steps, target_index)

    test_hours   = pd.date_range(start= test.start_date, end = test.end_date, freq = "1h", inclusive = "left")[input_steps:].hour.values
    Y_pred_baseline = np.array([historical_avg[h] for h in test_hours])
    r2_baseline = r2_score(Y_test.flatten(), Y_pred_baseline.flatten())
    mae_baseline = mean_absolute_error(Y_test.flatten(), Y_pred_baseline.flatten())
    mse_baseline = mean_squared_error(Y_test.flatten(), Y_pred_baseline.flatten())

    rmse_baseline = np.sqrt(mse_baseline)
    y_pred_last_step = X_test[:, -1, :, 0]
    r2_last = r2_score(Y_test,y_pred_last_step)
    mae_last = mean_absolute_error(Y_test,y_pred_last_step)
    mse_last = mean_squared_error(Y_test,y_pred_last_step)

    y_pred = dense_model.predict(data_test[:,:,1:], batch_size= 32)
    mse_d = mean_squared_error( y_pred.flatten(), data_test[:,:,0].flatten())
    mae_d = mean_absolute_error( y_pred.flatten(), data_test[:,:,0].flatten())
    r2_d = r2_score( y_pred.flatten(), data_test[:,:,0].flatten())

    y_pred = rnn_model.predict(X_test, batch_size= 32)
    mse_rnn = mean_squared_error( y_pred.flatten(), Y_test.flatten())
    mae_rnn = mean_absolute_error( y_pred.flatten(), Y_test.flatten())
    r2_rnn = r2_score( y_pred.flatten(), Y_test.flatten())

    y_pred = best_model.predict(X_test, batch_size=32)
    mse_model = mean_squared_error(y_pred.flatten(), Y_test.flatten())
    mae_model = mean_absolute_error(y_pred.flatten(), Y_test.flatten())
    r2_model = r2_score(y_pred.flatten(), Y_test.flatten())

    print("--- Average per zone per hour ---")
    print(f"\tR2: {r2_baseline:.4f}")
    print(f"\tMSE: {mse_baseline:.4f}")
    print(f"\tMAE: {mae_baseline:.4f}")
    print(f"\tRMSE: {rmse_baseline:.4f}\n")
    print("--- Always last step ---")
    print(f"\tR2: {r2_last:.4f}")
    print(f"\tMSE: {mse_last:.4f}")
    print(f"\tMAE: {mae_last:.4f}\n")
    print("--- Dense ---")
    print(f"\tR2: {r2_d:.4f}")
    print(f"\tMSE: {mse_d:.4f}")
    print(f"\tMAE: {mae_d:.4f}")
    print("--- SimpleRnn ---")
    print(f"\tR2: {r2_rnn:.4f}")
    print(f"\tMSE: {mse_rnn:.4f}")
    print(f"\tMAE: {mae_rnn:.4f}\n")
    print("--- STGCN model ---")
    print(f"\tR2: {r2_model:.4f}")
    print(f"\tMSE: {mse_model}")
    print(f"\tMAE: {mae_model}\n")

In [23]:
result(2025,10,2026,1)

69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
69/69 ━━━━━━━━━━━━━━━━━━━━ 4s 56ms/step
--- Average per zone per hour ---
	R2: 0.7976
	MSE: 3031.9666
	MAE: 27.4469
	RMSE: 55.0633

--- Always last step ---
	R2: 0.6005
	MSE: 1487.4252
	MAE: 21.4326

--- Dense ---
	R2: -4.7501
	MSE: 18788.6094
	MAE: 93.5444
--- SimpleRnn ---
	R2: 0.1869
	MSE: 55015.7852
	MAE: 125.3513

--- STGCN model ---
	R2: 0.9698
	MSE: 422.52130126953125
	MAE: 11.732110977172852



In [11]:
result(2024,1,2025,1)

275/275 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
275/275 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
275/275 ━━━━━━━━━━━━━━━━━━━━ 15s 55ms/step
--- Average per zone per hour ---
	R2: 0.8495
	MSE: 448.9928
	MAE: 6.8478
	RMSE: 21.1894

--- Always last step ---
	R2: -0.2604
	MSE: 245.2823
	MAE: 4.7550

--- Dense ---
	R2: 0.9241
	MSE: 247.5496
	MAE: 4.6854
--- SimpleRnn ---
	R2: 0.9421
	MSE: 161.0526
	MAE: 3.8603

--- STGCN model ---
	R2: 0.9624
	MSE: 107.34938049316406
	MAE: 3.278775930404663



In [29]:
result(2023,1,2024,1)

274/274 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
274/274 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
274/274 ━━━━━━━━━━━━━━━━━━━━ 15s 54ms/step
--- Average per zone per hour ---
	R2: 0.8255
	MSE: 458.9045
	MAE: 7.0668
	RMSE: 21.4221

--- Always last step ---
	R2: -0.3745
	MSE: 198.0296
	MAE: 4.1636

--- Dense ---
	R2: 0.9117
	MSE: 287.7464
	MAE: 4.7982
--- SimpleRnn ---
	R2: 0.9403
	MSE: 146.2372
	MAE: 3.5832

--- STGCN model ---
	R2: 0.9568
	MSE: 107.80174255371094
	MAE: 3.0680277347564697



In [12]:
result(2023,1,2024,1)

274/274 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
274/274 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
274/274 ━━━━━━━━━━━━━━━━━━━━ 15s 55ms/step
--- Average per zone per hour ---
	R2: 0.8255
	MSE: 458.9045
	MAE: 7.0668
	RMSE: 21.4221

--- Always last step ---
	R2: -0.3745
	MSE: 198.0296
	MAE: 4.1636

--- Dense ---
	R2: 0.9117
	MSE: 287.7464
	MAE: 4.7982
--- SimpleRnn ---
	R2: 0.9403
	MSE: 146.2372
	MAE: 3.5832

--- STGCN model ---
	R2: 0.9554
	MSE: 112.43736267089844
	MAE: 3.243720769882202



# Evaluación

In [30]:
test_dates = dates(2025,10,2026,1)
test = get_taxiDemand(test_dates)
data_test = test.prepare_dataset(feature_list)
#data_test = data_train
X_test, Y_test = test.construct_dataset(data_test, input_steps, target_index)

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

def evaluate_autoregressive_predictions(
    model, 
    initial_history, 
    true_future, 
    future_features, 
    horizon, 
    target_feat_idx=0
):
    # 1. Normalizamos el estado interno a (Time, Nodes, Features)
    if len(initial_history.shape) == 4:
        current_history = initial_history[0].copy()
    else:
        current_history = initial_history.copy()
        
    T, N, F = current_history.shape
    predictions = []
    
    for step in range(horizon):
        
       
        model_input = np.expand_dims(current_history, axis=0) # (1, T, N, F)

        # --- PREDICCIÓN ---
        pred = model.predict(model_input, verbose=0)
        
        # STGCN devuelve (1, N, 1).  RNN devuelve (1, N). 
        # Lo forzamos todo a (N, 1) para estandarizar
        pred_standard = pred.reshape((N, 1))
        
        predictions.append(pred_standard)
        
        # --- COMBINAR CON FUTURO ---
        next_step_features = future_features[step].copy()  # (N, F)
        
        # Sobrescribimos la columna objetivo (pickups) con nuestra predicción
        next_step_features[:, target_feat_idx] = pred_standard[:, 0]
        
        # --- ACTUALIZAR VENTANA ---
        next_step_expanded = np.expand_dims(next_step_features, axis=0) # (1, N, F)
        # Tiramos el paso más antiguo [1:, :, :] y metemos el nuevo
        current_history = np.concatenate([current_history[1:, :, :], next_step_expanded], axis=0)
        
    # --- MÉTRICAS FINALES ---
    predictions = np.stack(predictions, axis=0) # (horizon, N, 1)
    
    preds_rounded = np.round(predictions).astype(int)
    preds_flat = preds_rounded.flatten()
    true_flat = true_future.flatten()
    
    mse = mean_squared_error(preds_flat, true_flat)
    mae = mean_absolute_error(preds_flat, true_flat)
    
    preds_2d = preds_rounded.reshape((horizon, N))
    true_2d = true_future.reshape((horizon, N))
    cumulative_error = np.mean(np.abs(np.sum(preds_2d - true_2d, axis=0)))
    
    metrics = {
        "Autoregressive MSE": mse,
        "Autoregressive MAE": mae,
        "Absolute Cumulative Error": cumulative_error
    }
    
    return predictions, metrics

In [ ]:
import numpy as np
from tqdm import tqdm # Recomendado para ver el progreso: pip install tqdm

def evaluate_autoregressive_dataset(
    model, 
    data_test, 
    input_steps, 
    horizon, 
    model_type="stgcn", 
    target_feat_idx=0,
    step_size=1
):
    """
    Desliza una ventana por todo el dataset de prueba para evaluar predicciones autoregresivas.
    
    Args:
        model: El modelo Keras entrenado.
        data_test: Array completo de pruebas con forma (Total_Time, Nodes, Features).
        input_steps: Tamaño de la ventana histórica (ej: 12).
        horizon: Cuántas horas hacia el futuro predecir en cada paso.
        model_type: "stgcn", "rnn" o "dense".
        target_feat_idx: Índice de la variable a predecir (0 para pickups).
        step_size: Salto entre evaluaciones. 1 = evalúa todos los timestamps posibles.
                   (Si tarda mucho, súbelo a 12 o 24).
                   
    Returns:
        avg_metrics: Diccionario con la media de las métricas.
    """
    total_time = len(data_test)
    
    # El índice máximo donde podemos empezar a predecir sin salirnos del dataset
    max_start_idx = total_time - input_steps - horizon
    
    all_mse = []
    all_mae = []
    all_cum_error = []
    
    print(f"Iniciando evaluación autoregresiva ({model_type.upper()}). Horizonte: {horizon}h.")
    
    # Bucle por todos los instantes de tiempo posibles
    for start_idx in tqdm(range(0, max_start_idx + 1, step_size)):
        
        # 1. Extraer la ventana histórica (T, N, F)
        initial_history = data_test[start_idx : start_idx + input_steps]
        
        # 2. Extraer el futuro real objetivo (horizon, N)
        true_future = data_test[
            start_idx + input_steps : start_idx + input_steps + horizon, 
            :, 
            target_feat_idx
        ]
        
        # 3. Extraer las características futuras conocidas (horizon, N, F)
        future_features = data_test[
            start_idx + input_steps : start_idx + input_steps + horizon, 
            :, 
            :
        ]
        
        # 4. Llamar a la función que predice recursivamente
        _, metrics = evaluate_autoregressive_predictions(
            model=model,
            initial_history=initial_history,
            true_future=true_future,
            future_features=future_features,
            horizon=horizon,
            target_feat_idx=target_feat_idx
        )
        
        # 5. Acumular las métricas de esta ventana
        all_mse.append(metrics["Autoregressive MSE"])
        all_mae.append(metrics["Autoregressive MAE"])
        all_cum_error.append(metrics["Absolute Cumulative Error"])
        
    # Calcular la media final de todas las ventanas evaluadas
    avg_metrics = {
        "Mean Autoregressive MSE": np.mean(all_mse),
        "Mean Autoregressive MAE": np.mean(all_mae),
        "Mean Absolute Cumulative Error": np.mean(all_cum_error)
    }
    
    return avg_metrics

In [38]:
# Definimos cuántas horas a futuro queremos probar (ej. 24 horas = 1 día entero)
horizonte_prediccion = 2

# Ejecutamos la evaluación (para STGCN)
resultados_stgcn = evaluate_autoregressive_dataset(
    model=best_model,
    data_test=data_test,
    input_steps=12,
    horizon=horizonte_prediccion,
    model_type="stgcn",
    target_feat_idx=0,
    step_size=2  
)
resultados_rnn = evaluate_autoregressive_dataset(
    model=rnn_model,
    data_test=data_test,
    input_steps=12,
    horizon=horizonte_prediccion,
    model_type="rnn",
    target_feat_idx=0,
    step_size=2  
)

Iniciando evaluación autoregresiva (STGCN). Horizonte: 2h.


100%|██████████| 1098/1098 [00:10<00:00, 106.23it/s]


Iniciando evaluación autoregresiva (RNN). Horizonte: 2h.


100%|██████████| 1098/1098 [00:06<00:00, 177.12it/s]


In [39]:
print("\n--- Resultados Autoregresivos STGCN ---")
for k, v in resultados_stgcn.items():
    print(f"{k}: {v:.4f}")
print("\n--- Resultados Autoregresivos RNN ---")
for k, v in resultados_rnn.items():
    print(f"{k}: {v:.4f}")


--- Resultados Autoregresivos STGCN ---
Mean Autoregressive MSE: 165.3638
Mean Autoregressive MAE: 4.4873
Mean Absolute Cumulative Error: 7.8610

--- Resultados Autoregresivos RNN ---
Mean Autoregressive MSE: 296.3277
Mean Autoregressive MAE: 6.3865
Mean Absolute Cumulative Error: 11.7542


In [40]:
# Definimos cuántas horas a futuro queremos probar (ej. 24 horas = 1 día entero)
horizonte_prediccion = 4

# Ejecutamos la evaluación (para STGCN)
resultados_stgcn = evaluate_autoregressive_dataset(
    model=best_model,
    data_test=data_test,
    input_steps=12,
    horizon=horizonte_prediccion,
    model_type="stgcn",
    target_feat_idx=0,
    step_size=2  
)
resultados_rnn = evaluate_autoregressive_dataset(
    model=rnn_model,
    data_test=data_test,
    input_steps=12,
    horizon=horizonte_prediccion,
    model_type="rnn",
    target_feat_idx=0,
    step_size=2  
)
print("\n--- Resultados Autoregresivos STGCN ---")
for k, v in resultados_stgcn.items():
    print(f"{k}: {v:.4f}")
print("\n--- Resultados Autoregresivos RNN ---")
for k, v in resultados_rnn.items():
    print(f"{k}: {v:.4f}")

Iniciando evaluación autoregresiva (STGCN). Horizonte: 4h.


100%|██████████| 1097/1097 [00:19<00:00, 55.23it/s]


Iniciando evaluación autoregresiva (RNN). Horizonte: 4h.


100%|██████████| 1097/1097 [00:12<00:00, 90.49it/s]


--- Resultados Autoregresivos STGCN ---
Mean Autoregressive MSE: 277.8573
Mean Autoregressive MAE: 5.5887
Mean Absolute Cumulative Error: 18.5172

--- Resultados Autoregresivos RNN ---
Mean Autoregressive MSE: 373.0759
Mean Autoregressive MAE: 7.1628
Mean Absolute Cumulative Error: 26.3104


In [42]:
# Definimos cuántas horas a futuro queremos probar (ej. 24 horas = 1 día entero)
horizonte_prediccion = 8

# Ejecutamos la evaluación (para STGCN)
resultados_stgcn = evaluate_autoregressive_dataset(
    model=best_model,
    data_test=data_test,
    input_steps=12,
    horizon=horizonte_prediccion,
    model_type="stgcn",
    target_feat_idx=0,
    step_size=2  
)
resultados_rnn = evaluate_autoregressive_dataset(
    model=rnn_model,
    data_test=data_test,
    input_steps=12,
    horizon=horizonte_prediccion,
    model_type="rnn",
    target_feat_idx=0,
    step_size=2  
)
print("\n--- Resultados Autoregresivos STGCN ---")
for k, v in resultados_stgcn.items():
    print(f"{k}: {v:.4f}")
print("\n--- Resultados Autoregresivos RNN ---")
for k, v in resultados_rnn.items():
    print(f"{k}: {v:.4f}")

Iniciando evaluación autoregresiva (STGCN). Horizonte: 8h.


100%|██████████| 1095/1095 [00:39<00:00, 27.94it/s]


Iniciando evaluación autoregresiva (RNN). Horizonte: 8h.


100%|██████████| 1095/1095 [00:23<00:00, 46.49it/s]


--- Resultados Autoregresivos STGCN ---
Mean Autoregressive MSE: 457.5488
Mean Autoregressive MAE: 7.1440
Mean Absolute Cumulative Error: 46.2549

--- Resultados Autoregresivos RNN ---
Mean Autoregressive MSE: 488.6528
Mean Autoregressive MAE: 8.0268
Mean Absolute Cumulative Error: 59.4066


In [43]:
# Definimos cuántas horas a futuro queremos probar (ej. 24 horas = 1 día entero)
horizonte_prediccion = 12

# Ejecutamos la evaluación (para STGCN)
resultados_stgcn = evaluate_autoregressive_dataset(
    model=best_model,
    data_test=data_test,
    input_steps=12,
    horizon=horizonte_prediccion,
    model_type="stgcn",
    target_feat_idx=0,
    step_size=2  
)
resultados_rnn = evaluate_autoregressive_dataset(
    model=rnn_model,
    data_test=data_test,
    input_steps=12,
    horizon=horizonte_prediccion,
    model_type="rnn",
    target_feat_idx=0,
    step_size=2  
)
print("\n--- Resultados Autoregresivos STGCN ---")
for k, v in resultados_stgcn.items():
    print(f"{k}: {v:.4f}")
print("\n--- Resultados Autoregresivos RNN ---")
for k, v in resultados_rnn.items():
    print(f"{k}: {v:.4f}")

Iniciando evaluación autoregresiva (STGCN). Horizonte: 12h.


100%|██████████| 1093/1093 [00:58<00:00, 18.53it/s]


Iniciando evaluación autoregresiva (RNN). Horizonte: 12h.


100%|██████████| 1093/1093 [00:35<00:00, 30.89it/s]


--- Resultados Autoregresivos STGCN ---
Mean Autoregressive MSE: 604.1144
Mean Autoregressive MAE: 8.2536
Mean Absolute Cumulative Error: 78.6843

--- Resultados Autoregresivos RNN ---
Mean Autoregressive MSE: 565.8832
Mean Autoregressive MAE: 8.4858
Mean Absolute Cumulative Error: 94.1704


In [44]:
# Definimos cuántas horas a futuro queremos probar (ej. 24 horas = 1 día entero)
horizonte_prediccion = 16

# Ejecutamos la evaluación (para STGCN)
resultados_stgcn = evaluate_autoregressive_dataset(
    model=best_model,
    data_test=data_test,
    input_steps=12,
    horizon=horizonte_prediccion,
    model_type="stgcn",
    target_feat_idx=0,
    step_size=2  
)
resultados_rnn = evaluate_autoregressive_dataset(
    model=rnn_model,
    data_test=data_test,
    input_steps=12,
    horizon=horizonte_prediccion,
    model_type="rnn",
    target_feat_idx=0,
    step_size=2  
)
print("\n--- Resultados Autoregresivos STGCN ---")
for k, v in resultados_stgcn.items():
    print(f"{k}: {v:.4f}")
print("\n--- Resultados Autoregresivos RNN ---")
for k, v in resultados_rnn.items():
    print(f"{k}: {v:.4f}")

Iniciando evaluación autoregresiva (STGCN). Horizonte: 16h.


100%|██████████| 1091/1091 [01:17<00:00, 14.14it/s]


Iniciando evaluación autoregresiva (RNN). Horizonte: 16h.


100%|██████████| 1091/1091 [00:46<00:00, 23.39it/s]


--- Resultados Autoregresivos STGCN ---
Mean Autoregressive MSE: 730.8031
Mean Autoregressive MAE: 9.0424
Mean Absolute Cumulative Error: 112.4179

--- Resultados Autoregresivos RNN ---
Mean Autoregressive MSE: 619.8617
Mean Autoregressive MAE: 8.8276
Mean Absolute Cumulative Error: 130.2421


In [45]:
# Definimos cuántas horas a futuro queremos probar (ej. 24 horas = 1 día entero)
horizonte_prediccion = 24

# Ejecutamos la evaluación (para STGCN)
resultados_stgcn = evaluate_autoregressive_dataset(
    model=best_model,
    data_test=data_test,
    input_steps=12,
    horizon=horizonte_prediccion,
    model_type="stgcn",
    target_feat_idx=0,
    step_size=2  
)
resultados_rnn = evaluate_autoregressive_dataset(
    model=rnn_model,
    data_test=data_test,
    input_steps=12,
    horizon=horizonte_prediccion,
    model_type="rnn",
    target_feat_idx=0,
    step_size=2  
)
print("\n--- Resultados Autoregresivos STGCN ---")
for k, v in resultados_stgcn.items():
    print(f"{k}: {v:.4f}")
print("\n--- Resultados Autoregresivos RNN ---")
for k, v in resultados_rnn.items():
    print(f"{k}: {v:.4f}")

Iniciando evaluación autoregresiva (STGCN). Horizonte: 24h.


100%|██████████| 1087/1087 [01:56<00:00,  9.33it/s]


Iniciando evaluación autoregresiva (RNN). Horizonte: 24h.


100%|██████████| 1087/1087 [01:11<00:00, 15.26it/s]


--- Resultados Autoregresivos STGCN ---
Mean Autoregressive MSE: 938.8645
Mean Autoregressive MAE: 10.0407
Mean Absolute Cumulative Error: 180.4168

--- Resultados Autoregresivos RNN ---
Mean Autoregressive MSE: 678.3042
Mean Autoregressive MAE: 9.2510
Mean Absolute Cumulative Error: 203.6300
